# Automated Data Drift and Data Quality Monitoring Pipeline

This notebook builds the second automation solution. It only monitors **data drift** and **data quality**.

Model quality is intentionally out of this pipeline because it needs predictions and ground truth labels. Those inputs often arrive on a different schedule than the raw inference input data. Keeping this notebook focused makes it easier for a customer to plug in one current input file and one baseline file.

## Input file pattern

This notebook uses explicit S3 file locations by default:

- `baseline_data_s3_uri`: the reference CSV with headers
- `current_data_s3_uri`: the current CSV with headers

That is the recommended plug-and-play pattern because the caller controls exactly which file is monitored.

There is also an optional `use_latest_file_pickup` switch. If it is set to `true`, the processing script treats the S3 URI as a prefix and picks the newest CSV under that prefix. This can be useful for demos or simple scheduled jobs, but explicit file paths are safer when several files can land close together.


## Section 1: Install Packages and Import Libraries

The notebook uses the SageMaker Python SDK to define the pipeline and the standard AWS SDK (`boto3`) to create supporting resources such as SNS and EventBridge Scheduler.


In [ ]:
# Install the libraries used by this notebook.
# The processing script installs its own runtime libraries inside the processing job.
%pip install --no-cache-dir 'sagemaker>=3' mlflow==3.4.0 sagemaker-mlflow==0.2.0

In [ ]:
import boto3
import json
import time
from datetime import datetime

import sagemaker
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import ProcessingOutput, ProcessingS3Output
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep

In [ ]:
# Create AWS clients and SageMaker sessions.
# Session is used for normal SDK actions. PipelineSession is used when defining pipeline steps.
sagemaker_session = Session()
pipeline_session = PipelineSession()

region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = 'monitoring-pipeline'
timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')

sm_client = boto3.client('sagemaker', region_name=region)
sns_client = boto3.client('sns', region_name=region)
scheduler_client = boto3.client('scheduler', region_name=region)
iam_client = boto3.client('iam', region_name=region)
sts_client = boto3.client('sts', region_name=region)

print(f'Region: {region}')
print(f'Default S3 bucket: {bucket}')
print(f'SageMaker execution role: {role}')
print(f'Timestamp suffix: {timestamp_suffix}')

## Section 2: Configure the Monitoring Inputs

Update these values before running the notebook. The important change from the earlier design is that the pipeline receives the exact S3 file locations for monitoring. It does not need batch transform output or prediction files.


In [ ]:
# ============================================================
# UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================

# Baseline/reference data. This should be a CSV with headers and the feature
# distribution you expect in production. Training features are commonly used.
baseline_data_s3_uri = f's3://{bucket}/{prefix}/data/baseline/baseline.csv'

# Current data to monitor. This should also be a CSV with headers and the same
# feature columns as the baseline file. This is the file the customer wants to
# check for drift and quality issues.
current_data_s3_uri = f's3://{bucket}/{prefix}/data/current/current_features.csv'

# Recommended default: false. Use exact S3 file paths.
# Optional demo pattern: true. Treat the S3 values above as prefixes and pick
# the newest CSV under each prefix inside the processing job.
use_latest_file_pickup = 'false'

# If you ran Notebook 1 and have local demo data, set this to True to upload it.
# Keep it False when using customer-provided S3 files.
upload_demo_files = False
local_baseline_file = 'data/baseline.csv'
local_current_file = 'data/test_features_headers.csv'

# MLflow configuration. Use the same MLflow app and experiment you use for the
# training or experimentation notebook so monitoring runs are easy to compare.
mlflow_app_name = 'DefaultMLFlowApp'
mlflow_experiment_name = 'test-monitor-pipeline'
mlflow_run_name = f'data_drift_quality_monitoring_{timestamp_suffix}'

# SNS alerting. Replace this with your email address and confirm the email
# subscription after the SNS topic is created.
notification_email = '<ENTER>'

# EventBridge Scheduler expression. This is only used if you create the schedule
# in Section 8.
schedule_expression = 'rate(1 day)'

# Pipeline infrastructure.
pipeline_name = 'data-drift-quality-monitoring-pipeline'
instance_type = 'ml.m5.large'

print(f'Baseline data S3 URI     : {baseline_data_s3_uri}')
print(f'Current data S3 URI      : {current_data_s3_uri}')
print(f'Use latest file pickup   : {use_latest_file_pickup}')
print(f'MLflow app               : {mlflow_app_name}')
print(f'MLflow experiment        : {mlflow_experiment_name}')
print(f'Notification email       : {notification_email}')
print(f'Pipeline name            : {pipeline_name}')

In [ ]:
# Optional helper for the local demo data generated by Notebook 1.
# This cell is not needed when a customer already has files in S3.
if upload_demo_files:
    baseline_data_s3_uri = sagemaker_session.upload_data(
        path=local_baseline_file,
        bucket=bucket,
        key_prefix=f'{prefix}/data/baseline',
    )

    current_data_s3_uri = sagemaker_session.upload_data(
        path=local_current_file,
        bucket=bucket,
        key_prefix=f'{prefix}/data/current',
    )

    print(f'Uploaded baseline data to: {baseline_data_s3_uri}')
    print(f'Uploaded current data to : {current_data_s3_uri}')
else:
    print('Skipping demo upload. The pipeline will use the configured S3 URIs.')

## Section 3: Resolve the SageMaker Managed MLflow App ARN

SageMaker managed MLflow uses an app ARN as the tracking URI. This cell finds that ARN from the app name.


In [ ]:
mlflow_apps = sm_client.list_mlflow_apps()

mlflow_app_arn = None
for app in mlflow_apps['Summaries']:
    if app['Name'] == mlflow_app_name:
        mlflow_app_arn = app['Arn']
        break

if mlflow_app_arn is None:
    available_apps = [app['Name'] for app in mlflow_apps['Summaries']]
    raise ValueError(f'MLflow app {mlflow_app_name} not found. Available apps: {available_apps}')

print(f'MLflow app name: {mlflow_app_name}')
print(f'MLflow app ARN : {mlflow_app_arn}')

## Section 4: Create SNS Topic and Email Subscription

The processing script sends an alert only when Evidently detects drift. Data quality reports are still logged every run, whether or not drift is detected.


In [ ]:
topic_name = f'{pipeline_name}-drift-alerts'

create_topic_response = sns_client.create_topic(
    Name=topic_name,
    Tags=[
        {'Key': 'Project', 'Value': 'sagemaker-data-drift-quality-monitoring'},
        {'Key': 'ManagedBy', 'Value': 'batch-monitoring-notebook'},
    ],
)
sns_topic_arn = create_topic_response['TopicArn']
print(f'SNS topic ARN: {sns_topic_arn}')

if notification_email and notification_email != '<ENTER>':
    subscribe_response = sns_client.subscribe(
        TopicArn=sns_topic_arn,
        Protocol='email',
        Endpoint=notification_email,
    )
    print(f'Email subscription requested for: {notification_email}')
    print(f'Subscription ARN: {subscribe_response["SubscriptionArn"]}')
    print('Check your inbox and confirm the SNS subscription before expecting alerts.')
else:
    print('notification_email is not set. You can subscribe an email address later in the SNS console.')

## Section 5: Define the SageMaker Pipeline

This pipeline has one processing step. It does not run batch transform and it does not load predictions. That keeps Notebook 2 focused on data drift and data quality only.


In [ ]:
# Pipeline parameters make the data locations overridable at runtime.
# A scheduler, Lambda function, Step Functions workflow, or manual execution can
# pass a different current file without rebuilding the pipeline.
param_baseline_data_s3_uri = ParameterString(
    name='BaselineDataS3Uri',
    default_value=baseline_data_s3_uri,
)

param_current_data_s3_uri = ParameterString(
    name='CurrentDataS3Uri',
    default_value=current_data_s3_uri,
)

param_use_latest_file_pickup = ParameterString(
    name='UseLatestFilePickup',
    default_value=use_latest_file_pickup,
)

In [ ]:
# Use the standard SKLearn processing image. The script installs Evidently and
# MLflow at runtime so a custom image is not required for this sample.
sklearn_image_uri = image_uris.retrieve(
    framework='sklearn',
    region=region,
    version='1.2-1',
    py_version='py3',
    instance_type=instance_type,
)

monitoring_processor = ScriptProcessor(
    role=role,
    image_uri=sklearn_image_uri,
    command=['python3'],
    instance_count=1,
    instance_type=instance_type,
    volume_size_in_gb=30,
    max_runtime_in_seconds=3600,
    base_job_name='data-drift-quality-monitoring',
    sagemaker_session=pipeline_session,
    env={
        # The sagemaker-mlflow plugin reads this environment variable.
        'MLFLOW_TRACKING_URI': mlflow_app_arn,
    },
)

print(f'Processing image: {sklearn_image_uri}')

In [ ]:
# The processing job downloads the input CSV files itself from the explicit S3
# URIs. That makes the pipeline definition simple and avoids hardcoding file
# names in ProcessingInput channels.
processing_args = monitoring_processor.run(
    code='scripts/monitoring_processor.py',
    inputs=[],
    outputs=[
        ProcessingOutput(
            output_name='monitoring-reports',
            s3_output=ProcessingS3Output(
                s3_uri=f's3://{bucket}/{prefix}/monitoring-reports/data-drift-quality',
                local_path='/opt/ml/processing/output',
                s3_upload_mode='EndOfJob',
            ),
        ),
    ],
    arguments=[
        '--baseline-data-s3-uri', param_baseline_data_s3_uri,
        '--current-data-s3-uri', param_current_data_s3_uri,
        '--use-latest-file-pickup', param_use_latest_file_pickup,
        '--mlflow-tracking-uri', mlflow_app_arn,
        '--mlflow-experiment-name', mlflow_experiment_name,
        '--mlflow-run-name', mlflow_run_name,
        '--sns-topic-arn', sns_topic_arn,
        '--region', region,
    ],
)

processing_step = ProcessingStep(
    name='DataDriftAndQualityMonitoring',
    step_args=processing_args,
)

print('Processing step defined: DataDriftAndQualityMonitoring')

In [ ]:
monitoring_pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        param_baseline_data_s3_uri,
        param_current_data_s3_uri,
        param_use_latest_file_pickup,
    ],
    steps=[processing_step],
    sagemaker_session=pipeline_session,
)

monitoring_pipeline.upsert(role_arn=role)
print(f'Pipeline created or updated: {pipeline_name}')

## Section 6: Run a Test Execution

Run the pipeline once before scheduling it. This catches missing S3 permissions, bad file paths, schema mismatches, and MLflow configuration issues early.


In [ ]:
%%time
execution = monitoring_pipeline.start(
    parameters={
        'BaselineDataS3Uri': baseline_data_s3_uri,
        'CurrentDataS3Uri': current_data_s3_uri,
        'UseLatestFilePickup': use_latest_file_pickup,
    }
)

print(f'Pipeline execution started: {execution.arn}')
print('Waiting for the pipeline execution to finish...')
execution.wait()

status = execution.describe()['PipelineExecutionStatus']
print(f'Pipeline execution status: {status}')

In [ ]:
# Print each pipeline step status.
# If the processing job fails, the failure reason here usually points to the
# SageMaker Processing job logs in CloudWatch.
for step in execution.list_steps():
    print(f'Step: {step["StepName"]}')
    print(f'  Status: {step["StepStatus"]}')
    if 'FailureReason' in step:
        print(f'  Failure reason: {step["FailureReason"]}')

## Section 7: Review Results

Open SageMaker Studio and go to the MLflow app configured above. Look for the run named `data_drift_quality_monitoring_<timestamp>`.

The run contains:

- `DriftedColumnsCount.count` and `DriftedColumnsCount.share`
- current and baseline data quality metrics such as row count, missing cells, and duplicate rows
- Evidently HTML and JSON artifacts for data drift
- Evidently HTML and JSON artifacts for data quality
- `monitoring_summary.json` with the exact S3 files used


## Section 8: Schedule the Pipeline with EventBridge Scheduler

This section is optional. It creates a recurring schedule that starts the SageMaker Pipeline.

For production plug-and-play, many teams use an upstream orchestrator to pass the exact current file path. For a simple daily schedule, you can either keep explicit fixed file paths or set `use_latest_file_pickup = 'true'` and point the parameters at landing prefixes.


In [ ]:
pipeline_description = sm_client.describe_pipeline(PipelineName=pipeline_name)
pipeline_arn = pipeline_description['PipelineArn']
account_id = sts_client.get_caller_identity()['Account']

print(f'Pipeline ARN: {pipeline_arn}')
print(f'Account ID   : {account_id}')

In [ ]:
scheduler_role_name = f'{pipeline_name}-scheduler-role'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {'Service': 'scheduler.amazonaws.com'},
            'Action': 'sts:AssumeRole',
        }
    ],
}

try:
    iam_client.update_assume_role_policy(
        RoleName=scheduler_role_name,
        PolicyDocument=json.dumps(trust_policy),
    )
    print(f'Updated scheduler role trust policy: {scheduler_role_name}')
except iam_client.exceptions.NoSuchEntityException:
    iam_client.create_role(
        RoleName=scheduler_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Allows EventBridge Scheduler to start the data drift quality pipeline',
        Tags=[
            {'Key': 'Project', 'Value': 'sagemaker-data-drift-quality-monitoring'},
            {'Key': 'ManagedBy', 'Value': 'batch-monitoring-notebook'},
        ],
    )
    print(f'Created scheduler role: {scheduler_role_name}')

policy_document = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': 'sagemaker:StartPipelineExecution',
            'Resource': pipeline_arn,
        }
    ],
}

iam_client.put_role_policy(
    RoleName=scheduler_role_name,
    PolicyName='StartPipelineExecution',
    PolicyDocument=json.dumps(policy_document),
)

scheduler_role_arn = f'arn:aws:iam::{account_id}:role/{scheduler_role_name}'
print(f'Scheduler role ARN: {scheduler_role_arn}')

# IAM changes can take a few seconds to propagate.
time.sleep(10)

In [ ]:
schedule_name = f'{pipeline_name}-auto-schedule'

pipeline_parameters = [
    {'Name': 'BaselineDataS3Uri', 'Value': baseline_data_s3_uri},
    {'Name': 'CurrentDataS3Uri', 'Value': current_data_s3_uri},
    {'Name': 'UseLatestFilePickup', 'Value': use_latest_file_pickup},
]

schedule_target = {
    'Arn': pipeline_arn,
    'RoleArn': scheduler_role_arn,
    'SageMakerPipelineParameters': {
        'PipelineParameterList': pipeline_parameters,
    },
}

try:
    scheduler_client.update_schedule(
        Name=schedule_name,
        ScheduleExpression=schedule_expression,
        State='ENABLED',
        FlexibleTimeWindow={'Mode': 'OFF'},
        Target=schedule_target,
    )
    print(f'Updated schedule: {schedule_name}')
except scheduler_client.exceptions.ResourceNotFoundException:
    scheduler_client.create_schedule(
        Name=schedule_name,
        Description=f'Triggers {pipeline_name} on {schedule_expression}',
        ScheduleExpression=schedule_expression,
        State='ENABLED',
        FlexibleTimeWindow={'Mode': 'OFF'},
        Target=schedule_target,
    )
    print(f'Created schedule: {schedule_name}')

print(f'Schedule expression: {schedule_expression}')

In [ ]:
schedule_info = scheduler_client.get_schedule(Name=schedule_name)

print(f'Schedule: {schedule_info["Name"]}')
print(f'  State: {schedule_info["State"]}')
print(f'  Expression: {schedule_info["ScheduleExpression"]}')
print('  Pipeline parameters:')
for item in schedule_info['Target']['SageMakerPipelineParameters']['PipelineParameterList']:
    print(f'    {item["Name"]}: {item["Value"]}')

## Section 9: Cleanup

Run this section when you want to remove the resources created by this notebook. MLflow runs and S3 artifacts are preserved for history unless you delete them separately.


In [ ]:
try:
    scheduler_client.delete_schedule(Name=schedule_name)
    print(f'Deleted schedule: {schedule_name}')
except Exception as error:
    print(f'Schedule cleanup skipped or failed: {error}')

try:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print(f'Deleted pipeline: {pipeline_name}')
except Exception as error:
    print(f'Pipeline cleanup skipped or failed: {error}')

try:
    sns_client.delete_topic(TopicArn=sns_topic_arn)
    print(f'Deleted SNS topic: {sns_topic_arn}')
except Exception as error:
    print(f'SNS cleanup skipped or failed: {error}')

try:
    iam_client.delete_role_policy(RoleName=scheduler_role_name, PolicyName='StartPipelineExecution')
    iam_client.delete_role(RoleName=scheduler_role_name)
    print(f'Deleted scheduler role: {scheduler_role_name}')
except Exception as error:
    print(f'IAM cleanup skipped or failed: {error}')